In [2]:
# ============================================================
# TRANSFORM SPRINTS DATA — LOCAL EXECUTION
# ============================================================

import os
from pyspark.sql import functions as F
from pyspark.sql.types import *
import nbformat
from nbconvert import PythonExporter

In [3]:
# -----------------------------------------
# 1. Load environment + helpers
# -----------------------------------------
def run_notebook(path):
    with open(path, "r") as f:
        nb = nbformat.read(f, as_version=4)
    source, _ = PythonExporter().from_notebook_node(nb)
    exec(source, globals())

run_notebook("/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")
run_notebook("/Users/manoharazalki/F1-Analytics/notebooks/00-common/03_silver_helpers.ipynb")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/09 13:58:17 WARN Utils: Your hostname, Manohars-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.68.78.178 instead (on interface en0)
26/06/09 13:58:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/09 13:58:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/09 13:58:17 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/09 13:58:17 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/06/09 13:58:17 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/06/09 13:58:17 

===== F1 Analytics Environment Configuration =====
BASE_PATH:        /Users/manoharazalki/F1-Analytics/data
LANDING_PATH:     /Users/manoharazalki/F1-Analytics/data/landing
BRONZE_PATH:      /Users/manoharazalki/F1-Analytics/data/bronze
SILVER_PATH:      /Users/manoharazalki/F1-Analytics/data/silver
GOLD_PATH:        /Users/manoharazalki/F1-Analytics/data/gold
INCREMENTAL_PATH: /Users/manoharazalki/F1-Analytics/data/incremental
CONTROL_PATH:     /Users/manoharazalki/F1-Analytics/data/control


In [4]:
# -----------------------------------------
# 2. Receive p_batch_id (unified logic)
# -----------------------------------------

# Case 1: Papermill parameter
if "p_batch_id" in globals() and p_batch_id not in [None, ""]:
    pass

# Case 2: Databricks widget
elif "dbutils" in globals():
    p_batch_id = dbutils.widgets.get("p_batch_id")

# Case 3: Manual Jupyter execution → auto-detect latest batch
else:
    bronze_folder = f"{BRONZE_PATH}/sprints/data.parquet"
    batch_folders = [
        f.split("=")[1]
        for f in os.listdir(bronze_folder)
        if f.startswith("batch_id=")
    ]
    if not batch_folders:
        raise Exception("❌ No batch_id partitions found in Bronze sprints")
    p_batch_id = sorted(batch_folders)[-1]
    print("⚠️ Auto-selected latest batch_id:", p_batch_id)

# Final validation
if p_batch_id is None or p_batch_id == "":
    raise Exception("❌ p_batch_id not provided to Silver notebook")

print("Using p_batch_id:", p_batch_id)

⚠️ Auto-selected latest batch_id: 20260609_132357
Using p_batch_id: 20260609_132357


In [5]:
# -----------------------------------------
# 3. Define Bronze + Silver paths
# -----------------------------------------
bronze_path = f"{BRONZE_PATH}/sprints/data.parquet"
silver_path = f"{SILVER_PATH}/sprints"
os.makedirs(silver_path, exist_ok=True)

In [6]:
# -----------------------------------------
# 4. Read Bronze (ONLY this batch)
# -----------------------------------------
sprints_df = (
    spark.read.parquet(bronze_path)
    .filter(F.col("batch_id") == p_batch_id)
)

print("✔ Bronze sprints read")
sprints_df.show(5, truncate=False)

✔ Bronze sprints read
+----------+---------------------+-----+------+--------------------------------------------------------+-------------+--------------+----+----+------+------+--------+------------+--------+--------------------------+--------------------------------------------------------------+---------------+
|date      |raceName             |round|season|url                                                     |constructorId|driverId      |grid|laps|number|points|position|positionText|status  |ingest_timestamp          |source_file                                                   |batch_id       |
+----------+---------------------+-----+------+--------------------------------------------------------+-------------+--------------+----+----+------+------+--------+------------+--------+--------------------------+--------------------------------------------------------------+---------------+
|2023-04-30|azerbaijan grand prix|4    |2023  |https://en.wikipedia.org/wiki/2023_Azerbaijan_

In [7]:
# -----------------------------------------
# 5. Select required columns + standardize names
# -----------------------------------------
sprints_selected_df = (
    sprints_df
        .select(
            "season",
            "round",
            "constructorId",
            "driverId",
            "date",
            "raceName",
            "grid",
            "laps",
            "number",
            "points",
            "position",
            "positionText",
            "status",
            "ingest_timestamp",
            "source_file",
            "batch_id"
        )
        .withColumnsRenamed(
            {
                "constructorId": "constructor_id",
                "driverId": "driver_id",
                "raceName": "race_name",
                "date": "race_date",
                "grid": "grid_position",
                "laps": "completed_laps",
                "number": "car_number",
                "position": "final_position",
                "positionText": "final_position_text"
            }
        )
)

In [8]:
# -----------------------------------------
# 6. Business key validation + remove duplicates
# -----------------------------------------
sprints_valid_df = (
    sprints_selected_df
        .filter(
            F.col("season").isNotNull() &
            F.col("round").isNotNull() &
            F.col("constructor_id").isNotNull() &
            F.col("driver_id").isNotNull()
        )
        .dropDuplicates(["season", "round", "constructor_id", "driver_id"])
)

In [9]:
# -----------------------------------------
# 7. Title Case transformation
# -----------------------------------------
sprints_final_df = (
    sprints_valid_df
        .withColumn("race_name", F.initcap("race_name"))
)

In [10]:
# -----------------------------------------
# 8. Write to Silver
# -----------------------------------------
write_to_silver(
    sprints_final_df,
    f"{silver_path}/data.parquet",
    merge_keys=["season", "round", "constructor_id", "driver_id"]
)

print("✔ Sprints Silver written:", f"{silver_path}/data.parquet")

26/06/09 13:58:20 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


✔ Sprints Silver written: /Users/manoharazalki/F1-Analytics/data/silver/sprints/data.parquet


In [11]:
# -----------------------------------------
# 9. Validate
# -----------------------------------------
spark.read.parquet(f"{silver_path}/data.parquet").show(5, truncate=False)

+------+-----+--------------+----------+----------+------------------+-------------+--------------+----------+------+--------------+-------------------+--------+--------------------------+--------------------------------------------------------------+---------------+--------------------------+--------------------------+
|season|round|constructor_id|driver_id |race_date |race_name         |grid_position|completed_laps|car_number|points|final_position|final_position_text|status  |ingest_timestamp          |source_file                                                   |batch_id       |created_timestamp         |updated_timestamp         |
+------+-----+--------------+----------+----------+------------------+-------------+--------------+----------+------+--------------+-------------------+--------+--------------------------+--------------------------------------------------------------+---------------+--------------------------+--------------------------+
|2021  |10   |alfa          |giovi